# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dyajaballh8/FlyRank_Intern/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

## Method choice

I use Logistic Regression to predict whether a content item is a CTR improvement opportunity.

The target is:

`1` = high impressions, useful search position, and CTR below 0.5%.

`0` = otherwise.

Logistic Regression fits this task because the target is binary and the model produces probabilities that can be used to rank content items.

The model is compared with my Week-4 rule baseline on the same holdout split.

The features are observable search-performance signals:

- GSC impressions
- GSC clicks
- GSC average position

No product decision flags or future-window information are used.ne.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_auth (
    TYPE HTTP,
    BEARER_TOKEN '{HF_TOKEN}'
)
""")

DATA_PATH = """
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/data_0.parquet
""".replace("\n", "")

print("Connection ready.")


Connection ready.


## Split design

I use a grouped split by client.

Content from the same client can share similar traffic patterns and search behavior. A random row-level split could place rows from the same client in both training and test data, making evaluation too optimistic.

Therefore, clients in the test set are not used during training.

The model and the Week-4 baseline are evaluated on the same test clients.## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

data_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,

    CASE
        WHEN gsc_impressions >= 500
         AND gsc_avg_position > 0
         AND gsc_avg_position <= 20
         AND gsc_clicks * 100.0 / NULLIF(gsc_impressions, 0) < 0.5
        THEN 1
        ELSE 0
    END AS target

FROM read_parquet('{DATA_PATH}')

WHERE gsc_data_available = TRUE
  AND gsc_impressions > 0
  AND gsc_avg_position > 0
"""

df = con.execute(data_query).fetchdf()

print("Rows:", len(df))
print("Clients:", df["client_hash_id"].nunique())
print("\nTarget distribution:")
print(df["target"].value_counts())

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        groups=df["client_hash_id"]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("\nTrain rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

shared_clients = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])

print("Shared clients:", len(shared_clients))

assert len(shared_clients) == 0

print("\nGrouped split PASSED.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 4237020
Clients: 55

Target distribution:
target
0    4169358
1      67662
Name: count, dtype: int64

Train rows: 3336055
Test rows: 900965

Train clients: 44
Test clients: 11
Shared clients: 0

Grouped split PASSED.


## 3. Train + compare vs my baseline

## Train and compare with my baseline

The Week-4 baseline identifies CTR opportunities using fixed thresholds:

- impressions >= 500
- average position <= 20
- CTR < 0.5%

The Logistic Regression model uses the same observable search-performance information but learns how the signals combine.

Both approaches are evaluated on the same grouped test split.

The main comparison metric is F1 score because CTR opportunities are relatively uncommon and accuracy alone could be misleading.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

FEATURES = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

X_train = train_df[FEATURES]
y_train = train_df["target"]

X_test = test_df[FEATURES]
y_test = test_df["target"]

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

model_pred = model.predict(X_test)

test_df["ctr"] = (
    test_df["gsc_clicks"] * 100.0
    / test_df["gsc_impressions"]
)

baseline_pred = (
    (test_df["gsc_impressions"] >= 500)
    & (test_df["gsc_avg_position"] <= 20)
    & (test_df["ctr"] < 0.5)
).astype(int)

results = pd.DataFrame({
    "Method": [
        "Week-4 Rule Baseline",
        "Logistic Regression"
    ],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, model_pred)
    ],
    "Precision": [
        precision_score(y_test, baseline_pred, zero_division=0),
        precision_score(y_test, model_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, baseline_pred, zero_division=0),
        recall_score(y_test, model_pred, zero_division=0)
    ],
    "F1 Score": [
        f1_score(y_test, baseline_pred, zero_division=0),
        f1_score(y_test, model_pred, zero_division=0)
    ]
})

display(results.round(4))

print("\nLogistic Regression classification report:\n")

print(
    classification_report(
        y_test,
        model_pred,
        zero_division=0
    )
)


,Method,Accuracy,Precision,Recall,F1 Score
0,Week-4 Rule Baseline,1.0000,1.000,1.0000,1.0000
1,Logistic Regression,0.9867,0.493,0.9908,0.6584



Logistic Regression classification report:

              precision    recall  f1-score   support

           0       1.00      0.99      0.99    889271
           1       0.49      0.99      0.66     11694

    accuracy                           0.99    900965
   macro avg       0.75      0.99      0.83    900965
weighted avg       0.99      0.99      0.99    900965



## 4. Errors and interpretation

## Errors and interpretation

The model errors are reviewed using false positives and false negatives.

False positives are content items predicted as CTR opportunities when they do not meet the target definition.

False negatives are content items that meet the CTR opportunity definition but were missed by the model.

Feature coefficients are used as directional evidence of which search-performance signals the Logistic Regression model relies on.

These results are decision-support signals rather than proof that changing a title or snippet will improve CTR.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
test_df = test_df.copy()

test_df["actual"] = y_test
test_df["prediction"] = model_pred

test_df["prediction_probability"] = (
    model.predict_proba(X_test)[:, 1]
)

false_positives = test_df[
    (test_df["actual"] == 0)
    & (test_df["prediction"] == 1)
]

false_negatives = test_df[
    (test_df["actual"] == 1)
    & (test_df["prediction"] == 0)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nSample false positives:")
display(
    false_positives[
        FEATURES +
        [
            "actual",
            "prediction",
            "prediction_probability"
        ]
    ].head(10)
)

print("\nSample false negatives:")
display(
    false_negatives[
        FEATURES +
        [
            "actual",
            "prediction",
            "prediction_probability"
        ]
    ].head(10)
)


False positives: 11914
False negatives: 108

Sample false positives:


,gsc_impressions,gsc_clicks,gsc_avg_position,actual,prediction,prediction_probability
188,578,0,35.043253,0,1,0.735508
192,371,0,9.301887,0,1,0.662164
251,405,0,11.083951,0,1,0.771070
260,370,0,9.375676,0,1,0.652956
262,679,0,32.681885,0,1,0.984414
270,475,1,9.583158,0,1,0.790220
310,413,0,5.687651,0,1,0.923608
320,712,4,5.608146,0,1,0.906152
357,681,0,24.149780,0,1,0.997231
483,885,1,44.629379,0,1,0.994363



Sample false negatives:


,gsc_impressions,gsc_clicks,gsc_avg_position,actual,prediction,prediction_probability
27989,544,2,16.005515,1,0,0.461130
182233,502,2,19.073705,1,0,0.133865
185339,502,2,12.916335,1,0,0.343229
186976,524,2,15.730916,1,0,0.348112
390209,552,2,18.806159,1,0,0.377642
390417,605,3,14.400000,1,0,0.435405
390748,533,2,17.530957,1,0,0.321508
390809,563,2,17.998224,1,0,0.487384
488838,542,2,15.284133,1,0,0.483593
595918,619,3,17.744750,1,0,0.365055


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.